# Qwen CookingWorld capability checkpoint

Run the repository code on a Colab GPU. This notebook is a launcher, not a second implementation. Start with an L4-class 24 GB GPU if available; use FP16 explicitly for a T4. No training or quantization is performed.

**Before running:** commit and push the implementation, then enter its full Git SHA below. A private repository needs your normal secure GitHub access in Colab; do not paste tokens into this notebook. The public Qwen checkpoint ordinarily needs no token.

This checkpoint retains all prior visible observations/actions. It is not one of the three 256-token memory conditions. Start with one development game and inspect its trace before running more seeds. Real Qwen gameplay has not been validated by the offline tests.

The original v1 Colab notebook and its saved outputs are preserved at [commit 3addb54](https://github.com/sebace05/64610-Memory/blob/3addb547821991b79c9b11cd669e96e4a3e95aab/notebooks/qwen_colab.ipynb). This copy starts a separate v2 pilot.


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import uuid

REPO_URL = "https://github.com/sebace05/64610-Memory.git"
REPO_REF = "PASTE_COMMITTED_CODE_SHA"
REPO_DIR = Path("/content/64610-Memory-actor-v2")
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"  # Same checkpoint as the original development run.
DTYPE = "float16"      # T4-compatible; preserve precision for the v1/v2 comparison.
RECHECK_COST = 2
ACTION_BUDGET = 80
CONTEXT_LIMIT = 8192   # Overflow is an error; history is never silently truncated.
ACTION_MAX_NEW_TOKENS = 128
GENERATION_SEED = 0
ACTOR_PROMPT = "v2"    # Use "v1" for the original policy; create a separate pilot for each version.
FIRST_SEED = 0
MORE_SEEDS = [1, 2, 3, 4]  # Development only. Keep evaluation seeds separate.

## 1. Check out the selected source revision

An existing checkout is reused only if clean and already at the requested commit. No reset or overwrite is performed.

In [ ]:
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    raise ValueError("Set REPO_REF to the full pushed implementation commit SHA first.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True)
status = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
source_sha = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if status or source_sha != REPO_REF:
    raise RuntimeError("Existing checkout is dirty or at a different revision; preserve it and use a fresh REPO_DIR.")
print("Source commit:", source_sha)

## 2. Persist results before launching inference

Google Drive authorization happens through Colab. Every new launch directory preserves earlier attempts, including partial/error traces. Drive writes may add overhead to episode wall time; model latency is recorded separately.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PILOT_DIR = Path("/content/drive/MyDrive/64610-Memory/qwen-pilots") / uuid.uuid4().hex
PILOT_DIR.mkdir(parents=True, exist_ok=False)
print("Persistent pilot directory:", PILOT_DIR)

## 3. Install an isolated inference environment and run tests

The separate environment avoids conflicts with Colab's preinstalled Transformers, torchvision, or audio packages. This setup downloads the pinned CUDA-enabled PyTorch distribution; it may take several minutes. No model weights are loaded yet.

In [ ]:
import shutil

# Keep JVM container warnings off the stdout used by Py4J's port handshake.
# Set the parent environment too, so recreating ENV preserves this fix.
JAVA_GATEWAY_LOG_OPTIONS = "-Xlog:os+container=off:stdout -Xlog:os+container=warning:stderr"
java_options = os.environ.get("JAVA_TOOL_OPTIONS", "")
if JAVA_GATEWAY_LOG_OPTIONS not in java_options:
    java_options = (java_options + " " + JAVA_GATEWAY_LOG_OPTIONS).strip()
os.environ["JAVA_TOOL_OPTIONS"] = java_options

if shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "openjdk-17-jre-headless"], check=True)
subprocess.run(["java", "-version"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.11"], check=True)
runtime = Path("/content/memory-qwen-venv")
if not runtime.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", str(runtime)], check=True)
PYTHON = str(runtime / "bin/python")
ENV = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", HF_HOME="/content/hf-cache")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements-model.txt"], cwd=REPO_DIR, env=ENV, check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], cwd=REPO_DIR, env=ENV, check=True)

test_result = subprocess.run(
    [PYTHON, "-m", "pytest", "-q", "--tb=short"],
    cwd=REPO_DIR, env=ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
(PILOT_DIR / "pytest.log").write_text(test_result.stdout)
print(test_result.stdout)
print("Test log saved to:", PILOT_DIR / "pytest.log")
test_result.check_returncode()

(PILOT_DIR / "pip-freeze.txt").write_text(
    subprocess.check_output([PYTHON, "-m", "pip", "freeze"], env=ENV, text=True)
)


## 4. Verify the GPU and freeze the model revision

Use the selected precision explicitly; an unsupported GPU raises an error. No CPU fallback is used for this experiment. Copy the resolved model SHA into MODEL_REVISION for subsequent frozen runs.

In [ ]:
gpu_check = """
import json, torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime."
if {dtype!r} == "bfloat16":
    assert torch.cuda.is_bf16_supported(), "Select FP16 explicitly for this GPU."
print(json.dumps({{"gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "cuda": torch.version.cuda}}))
""".format(dtype=DTYPE)
gpu_info = json.loads(subprocess.check_output([PYTHON, "-c", gpu_check], env=ENV, text=True))
print(gpu_info)
if MODEL_REVISION is None:
    resolve = "from huggingface_hub import HfApi; print(HfApi().model_info({!r}).sha)".format(MODEL_ID)
    MODEL_REVISION = subprocess.check_output([PYTHON, "-c", resolve], env=ENV, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION):
    raise ValueError("Expected an immutable model commit SHA.")
manifest = dict(source_commit=source_sha, model_id=MODEL_ID, model_revision=MODEL_REVISION,
                dtype=DTYPE, gpu=gpu_info, recheck_cost=RECHECK_COST, action_budget=ACTION_BUDGET,
                context_limit=CONTEXT_LIMIT, action_max_new_tokens=ACTION_MAX_NEW_TOKENS,
                generation_seed=GENERATION_SEED, first_seed=FIRST_SEED, more_seeds=MORE_SEEDS,
                strategy="full_history", actor_prompt=ACTOR_PROMPT, diagnostic_only=True)
(PILOT_DIR / "pilot-manifest.json").write_text(json.dumps(manifest, indent=2))
print("Model commit:", MODEL_REVISION)

## 5. Run one real episode

Weights load once per invocation. A valid game loss returns exit code zero; model/context/infrastructure errors return nonzero and preserve their traces. Do not add constrained decoding or hidden history to improve this checkpoint without documenting the change.

ACTOR_PROMPT explicitly selects the policy version. v2 contains general planning and specific repair instructions; v1 retains the original behavior. Do not mix versions within a pilot directory. An improvement in actual Qwen completion remains to be measured.


In [ ]:
def run_diagnostic(seeds):
    recorded = json.loads((PILOT_DIR / "pilot-manifest.json").read_text())
    actual_source = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
    selected = dict(source_commit=actual_source, model_id=MODEL_ID, model_revision=MODEL_REVISION,
                    dtype=DTYPE, recheck_cost=RECHECK_COST, action_budget=ACTION_BUDGET,
                    context_limit=CONTEXT_LIMIT, action_max_new_tokens=ACTION_MAX_NEW_TOKENS,
                    generation_seed=GENERATION_SEED, actor_prompt=ACTOR_PROMPT)
    if any(recorded.get(key) != value for key, value in selected.items()):
        raise RuntimeError("Pilot settings changed. Start a separate pilot directory and manifest before running.")
    existing = [json.loads(p.read_text()) for p in PILOT_DIR.glob("*/result.json")]
    if any(r["config"]["seed"] in seeds for r in existing):
        raise RuntimeError("Requested seed already has a result. Preserve this attempt and start a new pilot directory to rerun.")
    if any(not (p.parent / "result.json").exists() for p in PILOT_DIR.glob("*/events.jsonl")):
        raise RuntimeError("This pilot contains a partial episode. Inspect it and use a separate directory for retries.")
    command = [PYTHON, "-m", "game_memory.run_episode", "--backend", "transformers",
               "--strategy", "full_history", "--actor-prompt", ACTOR_PROMPT, "--model-id", MODEL_ID, "--model-revision", MODEL_REVISION,
               "--dtype", DTYPE, "--context-limit", str(CONTEXT_LIMIT),
               "--action-max-new-tokens", str(ACTION_MAX_NEW_TOKENS),
               "--generation-seed", str(GENERATION_SEED), "--recheck-cost", str(RECHECK_COST),
               "--action-budget", str(ACTION_BUDGET), "--output-dir", str(PILOT_DIR),
               "--seeds", *map(str, seeds)]
    subprocess.run(command, cwd=REPO_DIR, env=ENV, check=True)

run_diagnostic([FIRST_SEED])

## 6. Audit the checkpoint and inspect the trace

The report independently checks budgets, tokens, request/response pairing, labels, and completion. It keeps infrastructure errors separate from gameplay losses. A technically valid report does not demonstrate adequate gameplay capability. Open the saved events.jsonl to inspect exact prompts and decisions.

In [ ]:
def audit_pilot():
    completed = subprocess.run([PYTHON, "-m", "game_memory.summarize_diagnostics", str(PILOT_DIR)],
                               cwd=REPO_DIR, env=ENV, text=True, capture_output=True)
    if completed.stdout:
        (PILOT_DIR / "checkpoint-report.json").write_text(completed.stdout)
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()

audit_pilot()

## Inspect actions and repair causes without rerunning the model

This reads saved logs only. It separates malformed JSON, invalid action schemas, and unavailable commands, and reports repeated actions, score plateaus, and recorded inventory-related commands. Score plateaus may include useful exploration; inspect observations before labeling them failures. These offline metrics never enter the agent prompt.

The first cell selects the episode for FIRST_SEED in this pilot and saves a JSON analysis and CSV table. You can instead set trace_path to an older pilot's events.jsonl to analyze the baseline.

In [ ]:
results = [json.loads(path.read_text()) for path in PILOT_DIR.glob("*/result.json")]
selected = [result for result in results if result["config"]["seed"] == FIRST_SEED]
if len(selected) != 1:
    raise RuntimeError("Expected exactly one result for FIRST_SEED; inspect the pilot directory.")
trace_path = PILOT_DIR / selected[0]["episode_id"] / "events.jsonl"
table_path = trace_path.parent / "actions.csv"
inspection = subprocess.run(
    [PYTHON, "-m", "game_memory.inspect_episode", str(trace_path), "--csv", str(table_path)],
    cwd=REPO_DIR, env=ENV, text=True, capture_output=True,
)
if inspection.stderr:
    print(inspection.stderr)
inspection.check_returncode()
analysis = json.loads(inspection.stdout)
(trace_path.parent / "trace-analysis.json").write_text(inspection.stdout)
print(json.dumps({key: value for key, value in analysis.items() if key != "turns"}, indent=2))
import pandas as pd
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(pd.read_csv(table_path, keep_default_na=False))


In [ ]:
STEP = 1  # Change to a repaired turn or the beginning of a repeated-action streak.
inspection = subprocess.run(
    [PYTHON, "-m", "game_memory.inspect_episode", str(trace_path), "--step", str(STEP), "--show-prompts"],
    cwd=REPO_DIR, env=ENV, text=True, capture_output=True,
)
if inspection.stderr:
    print(inspection.stderr)
inspection.check_returncode()
turn = json.loads(inspection.stdout)
for request in turn.get("requests", []):
    print("REQUEST:", request["purpose"])
    for message in request["messages"]:
        print(message["role"].upper() + ":\n" + message["content"])
print(json.dumps(turn["turns"], indent=2))


Optional: download the selected raw trace for review. This preserves the trace-download workflow from the saved Colab copy.

In [ ]:
from google.colab import files
files.download(str(trace_path))

## 7. Optional: run four more development games

Run this cell after inspecting the first episode. Keep settings unchanged. The four games share one model load. Look for evidence of valid gameplay, planning ability, and use of information already present in history. Five games provide a feasibility diagnostic, not a reliable success-rate estimate. Repeated full-history failure should be diagnosed before attributing bounded-memory losses to forgetting.

In [ ]:
run_diagnostic(MORE_SEEDS)
audit_pilot()

## Decision and next milestone

- Fix infrastructure or unexplained accounting failures before proceeding.
- Inspect formatting failures, output-limit hits, loops, navigation/cooking mistakes, and budget exhaustion.
- If capability looks promising, connect the implemented P1 prose/structured/hybrid writers to real inference and run the six-condition development pilot.
- If gameplay fails with history, use development data to diagnose capability or adjust task difficulty. Record every change and rerun consistently.
- Keep held-out evaluation seeds untouched. Do not interpret fewer rereads alone as better memory.

Results, the model/source revisions, dependency freeze, and reports are in PILOT_DIR. Disconnect the GPU runtime when finished. This notebook has not been executed on Colab as part of the CPU-only implementation validation.